In [ ]:
# 📈 Engagement Predictor from MovieLens
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load MovieLens ratings
df = pd.read_csv("../data/ratings.csv")

# Extract timestamp and compute daily watch count
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['timestamp'].dt.date

# Group: user engagement = number of ratings per user
engagement = df.groupby('userId').agg(
    total_ratings=('movieId', 'count'),
    avg_rating=('rating', 'mean'),
    active_days=('date', lambda x: len(set(x))),
)

# Simulate label: high engagement if total_ratings > median
median_ratings = engagement['total_ratings'].median()
engagement['high_engagement'] = (engagement['total_ratings'] > median_ratings).astype(int)

# Features and label
X = engagement.drop(columns=['high_engagement'])
y = engagement['high_engagement']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

# Save model
joblib.dump(model, "../backend/models/engagement_model.pkl")
print("✅ Saved to models/engagement_model.pkl")
